Cell 1 — Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

BASE_DIR = Path(r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting")

ADVANCED_DIR = BASE_DIR / "data" / "advanced"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
SUBMISSION_DIR = BASE_DIR / "submissions"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

Cell 2 — Load data

In [2]:
train = pd.read_csv(ADVANCED_DIR / "train_expanded_ewma_seasonal.csv")
test = pd.read_csv(PROCESSED_DIR / "test_features.csv")

train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

print(train.shape)
print(test.shape)
print(test["date"].min(), test["date"].max())

(3000888, 57)
(28512, 20)
2017-08-16 00:00:00 2017-08-31 00:00:00


Cell 3 — Feature list

In [3]:
target = "sales"

features = [
    "store_nbr", "family", "onpromotion",

    "year", "month", "day", "quarter", "week",
    "day_of_week", "is_weekend",

    "city", "state", "type", "cluster",

    "dcoilwtico",
    "is_payday",
    "is_national_holiday",
    "earthquake_period",

    "sales_lag_1",
    "sales_lag_2",
    "sales_lag_3",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_21",
    "sales_lag_28",
    "sales_lag_56",

    "rolling_mean_3",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_mean_56",

    "rolling_median_7",
    "rolling_median_14",
    "rolling_median_28",

    "rolling_min_7",
    "rolling_min_14",
    "rolling_min_28",

    "rolling_max_7",
    "rolling_max_14",
    "rolling_max_28",

    "rolling_std_7",
    "rolling_std_14",
    "rolling_std_28",
    "rolling_std_56",

    "promo_rolling_mean_7",
    "promo_rolling_mean_14",
    "promo_rolling_mean_28",

    "ewma_03",
    "ewma_05",
    "ewma_07",

    "sales_lag_364",
    "sales_lag_365",
    "sales_lag_371",
    "rolling_mean_364_7"
]

Cell 4 — Train final champion model

In [4]:
X = train[features]
y = np.log1p(train[target])

X = pd.get_dummies(X, drop_first=True)
train_columns = X.columns

final_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y)

print("Final champion model trained.")
print(X.shape)

Final champion model trained.
(3000888, 122)


Cell 5 — Prepare recursive history

In [5]:
history = train[
    ["date", "store_nbr", "family", "sales", "onpromotion"]
].copy()

predictions = []

lag_features = [
    "sales_lag_1", "sales_lag_2", "sales_lag_3",
    "sales_lag_7", "sales_lag_14", "sales_lag_21", "sales_lag_28", "sales_lag_56",
    "rolling_mean_3", "rolling_mean_7", "rolling_mean_14", "rolling_mean_28", "rolling_mean_56",
    "rolling_median_7", "rolling_median_14", "rolling_median_28",
    "rolling_min_7", "rolling_min_14", "rolling_min_28",
    "rolling_max_7", "rolling_max_14", "rolling_max_28",
    "rolling_std_7", "rolling_std_14", "rolling_std_28", "rolling_std_56",
    "promo_rolling_mean_7", "promo_rolling_mean_14", "promo_rolling_mean_28",
    "ewma_03", "ewma_05", "ewma_07",
    "sales_lag_364", "sales_lag_365", "sales_lag_371", "rolling_mean_364_7"
]

Cell 6 — Recursive prediction loop

In [6]:
for current_date in sorted(test["date"].unique()):

    print("Predicting:", current_date)

    current_test = test[test["date"] == current_date].copy()

    latest = (
        history.sort_values(["store_nbr", "family", "date"])
        .groupby(["store_nbr", "family"])
        .tail(371)
    )

    lag_table = (
        latest
        .groupby(["store_nbr", "family"])
        .agg(
            sales_lag_1=("sales", lambda x: x.iloc[-1] if len(x) >= 1 else 0),
            sales_lag_2=("sales", lambda x: x.iloc[-2] if len(x) >= 2 else 0),
            sales_lag_3=("sales", lambda x: x.iloc[-3] if len(x) >= 3 else 0),
            sales_lag_7=("sales", lambda x: x.iloc[-7] if len(x) >= 7 else 0),
            sales_lag_14=("sales", lambda x: x.iloc[-14] if len(x) >= 14 else 0),
            sales_lag_21=("sales", lambda x: x.iloc[-21] if len(x) >= 21 else 0),
            sales_lag_28=("sales", lambda x: x.iloc[-28] if len(x) >= 28 else 0),
            sales_lag_56=("sales", lambda x: x.iloc[-56] if len(x) >= 56 else 0),

            rolling_mean_3=("sales", lambda x: x.iloc[-3:].mean() if len(x) >= 3 else 0),
            rolling_mean_7=("sales", lambda x: x.iloc[-7:].mean() if len(x) >= 7 else 0),
            rolling_mean_14=("sales", lambda x: x.iloc[-14:].mean() if len(x) >= 14 else 0),
            rolling_mean_28=("sales", lambda x: x.iloc[-28:].mean() if len(x) >= 28 else 0),
            rolling_mean_56=("sales", lambda x: x.iloc[-56:].mean() if len(x) >= 56 else 0),

            rolling_median_7=("sales", lambda x: x.iloc[-7:].median() if len(x) >= 7 else 0),
            rolling_median_14=("sales", lambda x: x.iloc[-14:].median() if len(x) >= 14 else 0),
            rolling_median_28=("sales", lambda x: x.iloc[-28:].median() if len(x) >= 28 else 0),

            rolling_min_7=("sales", lambda x: x.iloc[-7:].min() if len(x) >= 7 else 0),
            rolling_min_14=("sales", lambda x: x.iloc[-14:].min() if len(x) >= 14 else 0),
            rolling_min_28=("sales", lambda x: x.iloc[-28:].min() if len(x) >= 28 else 0),

            rolling_max_7=("sales", lambda x: x.iloc[-7:].max() if len(x) >= 7 else 0),
            rolling_max_14=("sales", lambda x: x.iloc[-14:].max() if len(x) >= 14 else 0),
            rolling_max_28=("sales", lambda x: x.iloc[-28:].max() if len(x) >= 28 else 0),

            rolling_std_7=("sales", lambda x: x.iloc[-7:].std() if len(x) >= 7 else 0),
            rolling_std_14=("sales", lambda x: x.iloc[-14:].std() if len(x) >= 14 else 0),
            rolling_std_28=("sales", lambda x: x.iloc[-28:].std() if len(x) >= 28 else 0),
            rolling_std_56=("sales", lambda x: x.iloc[-56:].std() if len(x) >= 56 else 0),

            promo_rolling_mean_7=("onpromotion", lambda x: x.iloc[-7:].mean() if len(x) >= 7 else 0),
            promo_rolling_mean_14=("onpromotion", lambda x: x.iloc[-14:].mean() if len(x) >= 14 else 0),
            promo_rolling_mean_28=("onpromotion", lambda x: x.iloc[-28:].mean() if len(x) >= 28 else 0),

            ewma_03=("sales", lambda x: x.ewm(alpha=0.3).mean().iloc[-1] if len(x) >= 1 else 0),
            ewma_05=("sales", lambda x: x.ewm(alpha=0.5).mean().iloc[-1] if len(x) >= 1 else 0),
            ewma_07=("sales", lambda x: x.ewm(alpha=0.7).mean().iloc[-1] if len(x) >= 1 else 0),

            sales_lag_364=("sales", lambda x: x.iloc[-364] if len(x) >= 364 else 0),
            sales_lag_365=("sales", lambda x: x.iloc[-365] if len(x) >= 365 else 0),
            sales_lag_371=("sales", lambda x: x.iloc[-371] if len(x) >= 371 else 0),
            rolling_mean_364_7=("sales", lambda x: x.iloc[-371:-364].mean() if len(x) >= 371 else 0)
        )
        .reset_index()
    )

    current_test = current_test.merge(
        lag_table,
        on=["store_nbr", "family"],
        how="left"
    )

    current_test[lag_features] = current_test[lag_features].fillna(0)

    X_current = current_test[features]
    X_current = pd.get_dummies(X_current, drop_first=True)
    X_current = X_current.reindex(columns=train_columns, fill_value=0)

    log_preds = final_model.predict(X_current)

    preds = np.expm1(log_preds)
    preds = np.maximum(preds, 0)

    current_test["sales"] = preds

    predictions.append(
        current_test[["id", "sales"]]
    )

    new_history = current_test[
        ["date", "store_nbr", "family", "sales", "onpromotion"]
    ]

    history = pd.concat(
        [history, new_history],
        ignore_index=True
    )

Predicting: 2017-08-16 00:00:00
Predicting: 2017-08-17 00:00:00
Predicting: 2017-08-18 00:00:00
Predicting: 2017-08-19 00:00:00
Predicting: 2017-08-20 00:00:00
Predicting: 2017-08-21 00:00:00
Predicting: 2017-08-22 00:00:00
Predicting: 2017-08-23 00:00:00
Predicting: 2017-08-24 00:00:00
Predicting: 2017-08-25 00:00:00
Predicting: 2017-08-26 00:00:00
Predicting: 2017-08-27 00:00:00
Predicting: 2017-08-28 00:00:00
Predicting: 2017-08-29 00:00:00
Predicting: 2017-08-30 00:00:00
Predicting: 2017-08-31 00:00:00


Cell 7 — Save submission

In [7]:
submission = pd.concat(predictions, ignore_index=True)
submission = submission.sort_values("id")

submission_path = SUBMISSION_DIR / "submission_recursive_champion_xgboost.csv"

submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(submission.shape)
submission.head()

Saved: C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting\submissions\submission_recursive_champion_xgboost.csv
(28512, 2)


,id,sales
0,3000888,4.313488
1,3000889,0.021803
2,3000890,5.435405
3,3000891,2455.259766
4,3000892,0.045318
